# Tabelas Merge — Censo 2022 Castanhal

Notebook dedicado às tabelas consolidadas **por tópico** (base: doc TCC). Gera 5 DataFrames prontos para aplicação de ML:

1. **Demografia** — população, densidade, estrutura etária, razão de sexo, etc.
2. **Domicílios** — tipos, condições de moradia, ocupação
3. **Educação** — escolaridade, alfabetização, frequência escolar
4. **Trabalho e Renda** — PEA, taxa de atividade, profissões
5. **Renda** — per capita, distribuição de renda

**Fonte**: Google Drive `/content/drive/MyDrive/censo_castanhal/`

## 1. Setup

In [23]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/censo_castanhal'
    USAR_GITHUB = False
except ImportError:
    BASE = 'censo_castanhal/censo_castanhal'
    USAR_GITHUB = True  # local: carrega do GitHub se arquivo não existir

import pandas as pd
import numpy as np
import os
from urllib.parse import quote

GITHUB_RAW = 'https://raw.githubusercontent.com/LuanLindolfo/tcc/001-censo-streamlit-dashboard'
XLSX_PATH = 'censo_castanhal/censo_castanhal'

import io, requests

def _get_source(path):
    local = f'{BASE}/{path}'
    if os.path.exists(local):
        return local
    if USAR_GITHUB:
        url = f'{GITHUB_RAW}/{XLSX_PATH}/{quote(path)}'
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        return io.BytesIO(r.content)
    return local

def ler(path, **kwargs):
    return pd.read_excel(_get_source(path), **kwargs)

def q(df, col='Municípios', val='Castanhal'):
    return df.query(f"`{col}` == '{val}'") if col in df.columns else df

print(f'Base: {BASE}' + (' (GitHub fallback)' if USAR_GITHUB else ''))

Base: censo_castanhal (GitHub fallback)


## 2. Carregamento

In [24]:
# DEMOGRAFIA
populacao_residente = q(ler('Censo 2022 - População residente - Municípios.xlsx'))
densidade_populacional = q(ler('Censo 2022 - Densidade demográfica - Municípios.xlsx'))
populacao_urbana = q(ler('Censo 2022 - População urbana - Municípios.xlsx'))
taxa_crescimento = q(ler('Censo 2022 - Taxa de crescimento anual da população - Municípios.xlsx'))
piramide_etaria = ler('Censo 2022 - Pirâmide etária - Castanhal (PA).xlsx')
indice_envelhecimento_razao_sexo = ler('Censo 2022 - Razão de sexo e índice de envelhecimento - Castanhal (PA).xlsx')

# DOMICÍLIOS
taxa_domicilio = q(ler('Censo 2022 - Quantidade de domicílios - Municípios.xlsx'))
distribuicao_domicilios = ler('Censo 2022 - Tipos de domicílio - Castanhal (PA).xlsx')
numero_moradores = ler('Censo 2022 - Domicilios por numero de moradores - Castanhal (PA).xlsx')
media_moradores = ler('Censo 2022 - Média de moradores por domicílio - Castanhal (PA).xlsx')
caracteristicas_domicilio = ler('Censo 2022 - Características dos domicílios - Castanhal (PA).xlsx')
material_paredes = ler('Censo 2022 - Tipo de material das paredes externas - Castanhal (PA).xlsx')
condicoes_ocupacao = ler('Censo 2022 - Condições de ocupação - Castanhal (PA).xlsx')

# EDUCAÇÃO
nivel_instrucao = ler('Censo 2022 - Nível de instrução - Castanhal (PA).xlsx')
alfabetizacao = ler('Censo 2022 - Alfabetização - Castanhal (PA).xlsx')
escolaridade_mulheres_raw = ler('analise_escolaridade_mulheres.xlsx', header=None)
escolaridade_homens_raw = ler('analise_escolaridade_homens.xlsx', header=None)
escolaridade_mulheres_pct_raw = ler('analise_escolaridade_mulheres_percetual.xlsx', header=None)
escolaridade_homens_pct_raw = ler('analise_escolaridade_homens_percetual.xlsx', header=None)
frequencia_escolar_idade = ler('Censo 2022 - Taxa bruta de frequência escolar, por grupo de idade - Castanhal (PA).xlsx')

# TRABALHO E RENDA (PEA, profissões)
taxa_atividade_raw = ler('taxa_atividade.xlsx')
taxa_atividade_pct_raw = ler('taxa_atividade_percentual.xlsx')
profissao_raw = ler('profissões.xlsx')

# RENDA
rendimento_domiciliar_per_capita = ler('Censo 2022 - Rendimento domiciliar mensal per capita - Castanhal (PA).xlsx')
distribuicao_renda_raw = ler('distribuição de renda.xlsx')

print('Carregamento OK')

Carregamento OK


## 3. df_demografia

*Doc: População total e distribuição; Estrutura etária e de gênero; Étnico-racial (se disponível); Migração (se disponível)*

In [26]:
def std(df):
    d = df.copy()
    if 'Municípios' in d.columns: d.rename(columns={'Municípios': 'Municipio'}, inplace=True)
    if 'Município' in d.columns and 'Municipio' not in d.columns: d.rename(columns={'Município': 'Municipio'}, inplace=True)
    if 'Código municipal' in d.columns: d.rename(columns={'Código municipal': 'Codigo_Municipio'}, inplace=True)
    if 'Código do Município' in d.columns: d.rename(columns={'Código do Município': 'Codigo_Municipio'}, inplace=True)
    if 'Sigla UF' in d.columns: d.rename(columns={'Sigla UF': 'UF'}, inplace=True)
    return d

p = std(populacao_residente).rename(columns={'pessoas': 'Populacao_Total'})
p = pd.merge(p, std(densidade_populacional)[['Codigo_Municipio', 'habitantes por km²']].rename(columns={'habitantes por km²': 'Densidade_Demografica'}), on='Codigo_Municipio', how='left')
p = pd.merge(p, std(populacao_urbana)[['Codigo_Municipio', '%']].rename(columns={'%': 'Percentual_Urbano'}), on='Codigo_Municipio', how='left')
p = pd.merge(p, std(taxa_crescimento)[['Codigo_Municipio', '%']].rename(columns={'%': 'Taxa_Crescimento_Anual'}), on='Codigo_Municipio', how='left')

idx = indice_envelhecimento_razao_sexo[['pessoas com 60+ anos para cada 100 com até 14 anos', 'Município', 'Sigla UF', 'Código do Município']].copy()
idx = std(idx).rename(columns={'pessoas com 60+ anos para cada 100 com até 14 anos': 'Indice_Envelhecimento'})
raz = indice_envelhecimento_razao_sexo[['homens para cada 100 mulheres', 'Município', 'Sigla UF', 'Código do Município']].copy()
raz = std(raz).rename(columns={'homens para cada 100 mulheres': 'Razao_Sexo'})

p = pd.merge(p, idx[['Codigo_Municipio', 'Indice_Envelhecimento']], on='Codigo_Municipio', how='left')
p = pd.merge(p, raz[['Codigo_Municipio', 'Razao_Sexo']], on='Codigo_Municipio', how='left')

df_demografia = p
df_demografia

,Municipio,Codigo_Municipio,UF,Populacao_Total,Densidade_Demografica,Percentual_Urbano,Taxa_Crescimento_Anual,Indice_Envelhecimento,Razao_Sexo
0,Castanhal,1502400,PA,192256,"186,78","92,52","0,88","49,84","92,74"


In [27]:
# Pirâmide etária (estrutura por idade) — tabela complementar
piramide_etaria

,Grupo de idade,População feminina(pessoas),População masculina(pessoas),Município,Sigla UF,Código do Município,codMun
0,100 anos ou mais,21,8,Total,Castanhal,PA,1502400
1,95 a 99 anos,70,29,Total,Castanhal,PA,1502400
2,90 a 94 anos,194,111,Total,Castanhal,PA,1502400
3,85 a 89 anos,429,294,Total,Castanhal,PA,1502400
4,80 a 84 anos,868,615,Total,Castanhal,PA,1502400
5,75 a 79 anos,1346,1017,Total,Castanhal,PA,1502400
6,70 a 74 anos,2040,1640,Total,Castanhal,PA,1502400
7,65 a 69 anos,2911,2434,Total,Castanhal,PA,1502400
8,60 a 64 anos,3812,3177,Total,Castanhal,PA,1502400
9,55 a 59 anos,4599,4072,Total,Castanhal,PA,1502400


## 4. df_domicilios

*Doc: Tipos de domicílio; Condições de moradia (água, esgoto, lixo, energia, paredes); Posse e ocupação*

In [28]:
td = std(taxa_domicilio).rename(columns={'domicílios': 'Total_Domicilios'})
td = pd.merge(td, std(media_moradores)[['Codigo_Municipio', 'Média de moradores']].rename(columns={'Média de moradores': 'Media_Moradores_por_Domicilio'}), on='Codigo_Municipio', how='left')

df_domicilios = td

tipos = std(distribuicao_domicilios)
tipos_cols = [c for c in tipos.columns if c != 'Codigo_Municipio' and tipos[c].dtype in ['int64','float64']]
if tipos_cols:
    df_domicilios = pd.merge(df_domicilios, tipos[['Codigo_Municipio'] + tipos_cols], on='Codigo_Municipio', how='left', suffixes=('',''))

df_domicilios

,Municipio,Codigo_Municipio,UF,Total_Domicilios,Media_Moradores_por_Domicilio,Casa,Casa de vila ou condominio,Apartamento,Cortiço,Estrutura degradada ou inacabada
0,Castanhal,1502400,PA,80009,"3,01",54981,5126,3609,59,8


In [29]:
# Tabelas complementares (estruturas diferentes)
print('Características dos domicílios:'); display(caracteristicas_domicilio.head())
print('Material paredes:'); display(material_paredes.head())
print('Condições de ocupação:'); display(condicoes_ocupacao.head())

Características dos domicílios:


,Característica do domicílio,Total,Porcentagem,Município,Sigla UF,Código do Município
0,Conexão à rede de esgoto,10962,"17,19",Castanhal,PA,1502400
1,Abastecimento de água pela rede geral,27565,"43,22",Castanhal,PA,1502400
2,Banheiro de uso exclusivo,62782,"98,43",Castanhal,PA,1502400
3,Coleta de lixo,60413,"94,72",Castanhal,PA,1502400


Material paredes:


,Alvenaria ou taipa c/ revestimento,Alvenaria sem revestimento,Taipa sem revestimento,Madeira para construção,Madeira reaproveitada,Outro material,Sem parede,Município,Sigla UF,Código do Município
0,54379,5125,483,667,155,3008,0,Castanhal,PA,1502400


Condições de ocupação:


,Condição,Domicílios,Porcentagem,Município,Sigla UF,Código do Município
0,"Próprio, já pago",40030,"62,73",Castanhal,PA,1502400
1,"Próprio, pagando",7877,"12,34",Castanhal,PA,1502400
2,Alugado,11685,"18,31",Castanhal,PA,1502400
3,Cedido ou emprestado,3175,"4,98",Castanhal,PA,1502400
4,Outro,1051,"1,65",Castanhal,PA,1502400


## 5. df_educacao

*Doc: Nível de escolaridade; Alfabetização; Escolaridade por idade e sexo; Frequência escolar*

In [30]:
def extrair_esc(df_raw, sexo, pct=False):
    df = df_raw.copy() if isinstance(df_raw, pd.DataFrame) else ler(df_raw, header=None)
    # Estrutura real: col 1 = Sexo (Mulheres/Homens), row com dados tem valores em cols 3-7
    idx = df[df.iloc[:, 1].astype(str).str.strip() == sexo].index
    if len(idx) == 0: return pd.DataFrame()
    row = idx[0]
    muni = df.iloc[row, 0] if pd.notna(df.iloc[row, 0]) else 'Castanhal (PA)'
    t = lambda x: pd.to_numeric(x, errors='coerce')
    cols = ['Total_Pessoas_18_ou_mais','Sem_instr_fund_incompl','Fund_compl_medio_incompl','Medio_compl_super_incompl','Superior_completo']
    if pct: cols = [c+'_Perc' for c in cols]
    return pd.DataFrame({
        'Municipio': [muni], 'Sexo': [sexo], 'Ano': [2022],
        cols[0]: [t(df.iloc[row, 3])], cols[1]: [t(df.iloc[row, 4])], cols[2]: [t(df.iloc[row, 5])],
        cols[3]: [t(df.iloc[row, 6])], cols[4]: [t(df.iloc[row, 7])]
    })

em = extrair_esc(escolaridade_mulheres_raw, 'Mulheres')
eh = extrair_esc(escolaridade_homens_raw, 'Homens')
emp = extrair_esc(escolaridade_mulheres_pct_raw, 'Mulheres', pct=True)
ehp = extrair_esc(escolaridade_homens_pct_raw, 'Homens', pct=True)

df_educacao_sexo = pd.concat([em, eh], ignore_index=True)
df_educacao_sexo_pct = pd.concat([emp, ehp], ignore_index=True)

df_educacao = df_educacao_sexo.copy()
print('df_educacao:'); display(df_educacao)
print('df_educacao_sexo_pct:'); display(df_educacao_sexo_pct)
print('Complementares: nivel_instrucao, alfabetizacao, frequencia_escolar_idade')

df_educacao:


""


df_educacao_sexo_pct:


""


Complementares: nivel_instrucao, alfabetizacao, frequencia_escolar_idade


## 6. df_trabalho_renda

*Doc: PEA, taxa de atividade; Distribuição por setor; Tipos de ocupação (profissões)*

In [31]:
def limpar_taxa_atividade(df_raw):
    df = ler('taxa_atividade.xlsx', header=None) if not isinstance(df_raw, pd.DataFrame) else df_raw.copy()
    idx = df[df.iloc[:, 1].astype(str).str.contains('Economicamente ativas', na=False)].index
    if len(idx) == 0: return pd.DataFrame()
    rows = df.iloc[idx[0]:idx[0]+4, [1, 2]].copy()
    rows.columns = ['Condicao_Atividade', 'Total']
    rows['Total'] = pd.to_numeric(rows['Total'], errors='coerce').fillna(0).astype(int)
    rows['Codigo_Municipio'] = 1502400
    return rows

taxa_atividade = limpar_taxa_atividade(taxa_atividade_raw).reset_index(drop=True)

def limpar_taxa_atividade_percentual(_):
    df = ler('taxa_atividade_percentual.xlsx', header=None)
    idx = df[df.iloc[:, 1].astype(str).str.contains('Ano x Seção de atividade do trabalho principal', na=False)].index
    if len(idx) == 0: return pd.DataFrame()
    header_idx = idx[0]
    col_names = df.iloc[header_idx + 2, 2:].tolist()
    values = df.iloc[header_idx + 3, 2:].tolist()
    def to_val(v):
        res = pd.to_numeric(str(v).replace(',', '.'), errors='coerce')
        return res if not pd.isna(res) else 0.0
    out = pd.DataFrame({'Seção de Atividade': col_names, 'Valor': [to_val(v) for v in values]})
    out = out.dropna(subset=['Seção de Atividade'])
    col_str = out['Seção de Atividade'].astype(str).str.strip()
    out = out[(col_str != '') & (col_str.str.lower() != 'nan') & (col_str != 'Total')]
    return out.reset_index(drop=True)

taxa_atividade_pct_cleaned = limpar_taxa_atividade_percentual(None)

df_trabalho_renda = taxa_atividade.copy()

def limpar_profissoes(_):
    df = ler('profissões.xlsx', header=None)
    header_key = 'Ano x Grandes grupos de ocupação no trabalho principal'
    idx = df[df.iloc[:, 1].astype(str).str.contains(header_key, na=False)].index
    if len(idx) == 0: return pd.DataFrame()
    header_idx = idx[0]
    col_names = df.iloc[header_idx + 2, 2:].tolist()
    values = df.iloc[header_idx + 3, 2:].tolist()
    def to_val(v):
        res = pd.to_numeric(v, errors='coerce')
        return int(res) if not pd.isna(res) else 0
    out = pd.DataFrame({'Grupo de Ocupação': col_names, 'Valor': [to_val(v) for v in values]})
    out = out.dropna(subset=['Grupo de Ocupação'])
    col_str = out['Grupo de Ocupação'].astype(str).str.strip()
    out = out[(col_str != '') & (col_str.str.lower() != 'nan') & (col_str != 'Total')]
    return out.reset_index(drop=True)

profissao_cleaned = limpar_profissoes(None)

print('df_trabalho_renda (PEA):'); display(df_trabalho_renda)
print('Taxa atividade por setor (%):'); display(taxa_atividade_pct_cleaned)
print('Profissões (limpo):'); display(profissao_cleaned)

df_trabalho_renda (PEA):


,Condicao_Atividade,Total,Codigo_Municipio
0,Economicamente ativas,75129,1502400
1,Economicamente ativas - ocupadas,67766,1502400
2,Economicamente ativas - desocupadas,7363,1502400
3,Não economicamente ativas,66705,1502400


Taxa atividade por setor (%):


,Seção de Atividade,Valor
0,"Agricultura, pecuária, produção florestal, pes...",5959.0
1,Indústrias extrativas,124.0
2,Indústrias de transformação,7646.0
3,Eletricidade e gás,285.0
4,"Água, esgoto, atividades de gestão de resíduos...",261.0
5,Construção,5778.0
6,Comércio; reparação de veículos automotores e ...,17502.0
7,"Transporte, armazenagem e correio",2998.0
8,Alojamento e alimentação,2239.0
9,Informação e comunicação,452.0


Profissões (limpo):


,Grupo de Ocupação,Valor
0,Diretores e gerentes,1927
1,Profissionais das ciências e intelectuais,4675
2,Técnicos e profissionais de nível médio,3866
3,Trabalhadores de apoio administrativo,4605
4,"Trabalhadores dos serviços, vendedores dos com...",14628
5,"Trabalhadores qualificados da agropecuária, fl...",3608
6,"Trabalhadores qualificados, operários e artesã...",9141
7,Operadores de instalações e máquinas e montadores,6167
8,Ocupações elementares,12878
9,"Membros das forças armadas, policiais, bombeir...",516


## 7. df_renda

*Doc: Renda média per capita; Distribuição de renda; Índice de Gini (se possível)*

In [32]:
def limpar_distribuicao_renda(_):
    df = ler('distribuição de renda.xlsx', header=None)
    start = df[df.iloc[:, 1].astype(str).str.contains('Até 1/4 de salário mínimo', na=False)].index[0]
    out = df.iloc[start:, [1, 2]].copy()
    out.columns = ['Classes_de_Rendimento', 'Total']
    out['Total'] = pd.to_numeric(out['Total'], errors='coerce').fillna(0).astype(int)
    out = out.dropna(subset=['Classes_de_Rendimento'])
    col_str = out['Classes_de_Rendimento'].astype(str).str.strip()
    out = out[(col_str != '') & (col_str.str.lower() != 'nan') & (~col_str.str.contains('Total', na=False))]
    return out.reset_index(drop=True)

distribuicao_renda = limpar_distribuicao_renda(distribuicao_renda_raw)

rend = std(rendimento_domiciliar_per_capita).rename(columns={'Rendimento (R$)': 'Rendimento_Per_Capita'})

df_renda = rend[['Codigo_Municipio', 'Municipio', 'Rendimento_Per_Capita']].copy()
df_renda['Total_Domicilios_DistRenda'] = distribuicao_renda['Total'].sum()

print('df_renda:'); display(df_renda)
print('distribuicao_renda (Classes):'); display(distribuicao_renda)

df_renda:


,Codigo_Municipio,Municipio,Rendimento_Per_Capita,Total_Domicilios_DistRenda
0,1502400,Castanhal,"1.055,05",75394


distribuicao_renda (Classes):


,Classes_de_Rendimento,Total
0,Até 1/4 de salário mínimo,3368
1,Mais de 1/4 a 1/2 salário mínimo,6848
2,Mais de 1/2 a 1 salário mínimo,26202
3,Mais de 1 a 2 salários mínimos,22647
4,Mais de 2 a 3 salários mínimos,7647
5,Mais de 3 a 5 salários mínimos,4775
6,Mais de 5 a 10 salários mínimos,2146
7,Mais de 10 a 15 salários mínimos,340
8,Mais de 15 a 20 salários mínimos,299
9,Mais de 20 salários mínimos,243


## 8. Resumo — DataFrames por tópico

In [35]:
dfs = {
    'df_demografia': df_demografia,
    'df_domicilios': df_domicilios,
    'df_educacao': df_educacao,
    'df_educacao_sexo_pct': df_educacao_sexo_pct,
    'df_trabalho_renda': df_trabalho_renda,
    'taxa_atividade_pct_cleaned': taxa_atividade_pct_cleaned,
    'profissao_cleaned': profissao_cleaned,
    'df_renda': df_renda,
    'distribuicao_renda': distribuicao_renda,
    'piramide_etaria': piramide_etaria,
}
for nome, d in dfs.items():
    print(f'{nome}: {d.shape}')

df_demografia: (1, 9)
df_domicilios: (1, 10)
df_educacao: (0, 0)
df_educacao_sexo_pct: (0, 0)
df_trabalho_renda: (4, 3)
taxa_atividade_pct_cleaned: (22, 2)
profissao_cleaned: (11, 2)
df_renda: (1, 4)
distribuicao_renda: (11, 2)
piramide_etaria: (21, 7)
